# 03 — Data Preprocessing

## Objective
Prepare the dataset for machine learning without data leakage.

### Steps
1. Define features and target.
2. Split data into training and test sets using stratification.
3. Handle numerical and categorical features with a `ColumnTransformer`.
4. Save the fitted preprocessing pipeline.
5. Save train/test data for the next notebooks.

### Why use a pipeline?
The preprocessing steps must be fitted only on training data. This prevents information from the test set leaking into the model.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)

# Basic cleaning
df = df.drop_duplicates().reset_index(drop=True)

# Save a clean copy for documentation/reporting.
df.to_csv(PROCESSED_DIR / "cleaned_data.csv", index=False)

TARGET = "Failure"
X = df.drop(columns=[TARGET])
y = df[TARGET]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()

print("Features:", X.columns.tolist())
print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

Features: ['Temperature_C', 'Pressure_bar', 'Vibration_mm_s', 'Rotational_Speed_rpm', 'Voltage_V', 'Current_A', 'Operating_Hours', 'Tool_Wear_Min', 'Machine_Type', 'Maintenance_History_Count']
Categorical: ['Machine_Type']
Numerical: ['Temperature_C', 'Pressure_bar', 'Vibration_mm_s', 'Rotational_Speed_rpm', 'Voltage_V', 'Current_A', 'Operating_Hours', 'Tool_Wear_Min', 'Maintenance_History_Count']


In [2]:
# Stratified split keeps the failure/non-failure ratio approximately similar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training failure rate:", round(y_train.mean(), 4))
print("Test failure rate:", round(y_test.mean(), 4))

Training rows: 9600
Test rows: 2400
Training failure rate: 0.1306
Test failure rate: 0.1304


In [3]:
# Build preprocessing pipeline
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

# Fit ONLY on training data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (9600, 12)
Processed test shape: (2400, 12)


In [4]:
# Save datasets and preprocessing object
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

joblib.dump(preprocessor, MODEL_DIR / "preprocessor.pkl")

print("Saved:")
for path in [
    PROCESSED_DIR / "X_train.csv",
    PROCESSED_DIR / "X_test.csv",
    PROCESSED_DIR / "y_train.csv",
    PROCESSED_DIR / "y_test.csv",
    MODEL_DIR / "preprocessor.pkl",
]:
    print(" -", path)

Saved:
 - D:\Python_Programming\College Project\AI-Based Predictive Maintenance & Equipment Failure Detection System\ML_Predictive_Maintenance Code\data\processed\X_train.csv
 - D:\Python_Programming\College Project\AI-Based Predictive Maintenance & Equipment Failure Detection System\ML_Predictive_Maintenance Code\data\processed\X_test.csv
 - D:\Python_Programming\College Project\AI-Based Predictive Maintenance & Equipment Failure Detection System\ML_Predictive_Maintenance Code\data\processed\y_train.csv
 - D:\Python_Programming\College Project\AI-Based Predictive Maintenance & Equipment Failure Detection System\ML_Predictive_Maintenance Code\data\processed\y_test.csv
 - D:\Python_Programming\College Project\AI-Based Predictive Maintenance & Equipment Failure Detection System\ML_Predictive_Maintenance Code\models\preprocessor.pkl


## Preprocessing summary

- Numerical columns: median imputation + standardization.
- Categorical columns: most-frequent imputation + one-hot encoding.
- `Machine_Type` is converted into numerical indicator columns.
- The target is **not** scaled or encoded as a feature.
- Test data is transformed using the already-fitted training preprocessor.


## Preprocessing Design

### Step 1: Separate Features and Target
The input features are separated from the Failure target column.

### Step 2: Train-Test Split
The dataset is split into training and test sets using an 80:20
ratio and stratification to preserve the target-class proportion.

### Step 3: Numerical Features
Missing numerical values are handled using median imputation.
StandardScaler standardizes numerical feature values.

### Step 4: Categorical Features
Missing categorical values are handled using the most frequent
category. OneHotEncoder converts categories into numeric indicators.

### Step 5: Fit on Training Data
The preprocessing pipeline is fitted on training data only.
The same fitted transformations are applied to test data.

### Why This Matters
Fitting preprocessing on the full dataset can leak information
from the test set into model development. Keeping the test data
separate gives a more reliable estimate of generalization.
